In [ ]:
from src.config import DATA_ROOT, FIGURE_ROOT, SEQ_LENGTH
from pathlib import Path
for folder in ["", "shap", "embeddings"]:
    (Path(FIGURE_ROOT) / folder).mkdir(parents=True, exist_ok=True)


# 1. Basic Plot Functions

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

label_names = {
    "fall20": "Fall 20",
    "fall30": "Fall 30",
    "nadir90": "Nadir 90",
    "nadir100": "Nadir 100",
    "hemo": "HEMO",
    "kdoqi": "KDOQI"
}

def plot_auc_calibrate(true_y, y_prob, auc_ax=None, cali_ax=None, best_threshold=None, remark="", color="#F6A5B4", all_axs=None, legend=True, subtitle="", pr_ax=None):
    # roc curve, determine optimal threshold, and calibration curve
    fpr, tpr, thresholds = roc_curve(true_y, y_prob)
    precision, recall, _ = precision_recall_curve(true_y, y_prob)
    true_pos, pred_pos = calibration_curve(true_y, y_prob, n_bins=150)

    # Threshold variant: maximize Youden's J instead of minimizing distance to the top-left ROC corner.
    # best_threshold = thresholds[(tpr - fpr).argmax()]
    if best_threshold is None:
        distances = np.sqrt((fpr - 0)**2 + (tpr - 1)**2)
        optimal_idx = np.argmin(distances)
        best_threshold = thresholds[optimal_idx]
    
    # Threshold variant: use a fixed 0.5 cutoff instead of a validation-selected threshold.
    # confusion matrix
    # thres = 0.5
    thres = round(float(best_threshold), 2)
    y_pred = (np.array(y_prob) >= thres).astype(int)
    tn, fp, fn, tp = confusion_matrix(true_y, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    FPR = fp / (fp+tn)
    FNR = fn / (fn+tp)

    # log
    if auc_ax is not None:
        if all_axs is None:
            metrics = f"""{remark}
    AUC ROC: {round(roc_auc_score(true_y, y_prob), 3)}
    PR-AUC: {round(average_precision_score(true_y, y_prob), 3)}
    Specificity: {round(specificity, 3)}
    Sensitivity: {round(sensitivity, 3)}
    Balanced Acc: {round(0.5*(specificity+sensitivity), 3)}
    False Positive Rate: {round(FPR, 3)}
    False Negative Rate: {round(FNR, 3)}"""
            auc_ax.plot(fpr, tpr, label=metrics, color=color)
            auc_ax.legend(fontsize=10)
        else:
            auc_ax.plot(fpr, tpr, color=color, label=f"ROC Curve {remark}")
            if legend:
                auc_ax.legend(fontsize=10)
        auc_ax.plot([0, 1], [0, 1], 'k--', alpha=0.2, linewidth=1.0)
        auc_ax.set_xlabel('False Positive Rate', fontsize=12)
        auc_ax.set_ylabel('True Positive Rate', fontsize=12)
        auc_ax.set_title(f"{subtitle}: ROC Curve", fontsize=14)

    if pr_ax is not None:
        pr_ax.step(recall[::-1], precision[::-1], where='post', color=color, label=f"PR Curve {remark}")
        if legend:
            pr_ax.legend(fontsize=10)
        pr_ax.set_xlabel('Recall', fontsize=12)
        pr_ax.set_ylabel('Precision', fontsize=12)
        pr_ax.set_xlim(0, 1)
        pr_ax.set_ylim(0, 1)
        pr_ax.set_title(f"{subtitle}: Precision-Recall Curve", fontsize=14)

    if cali_ax is not None:
        # calibrate y_prob
        if all_axs is None:
            calibrate_metrics = f"""{remark}, Brier Score: {round(np.mean((true_pos-pred_pos)**2), 3)}"""
            cali_ax.scatter(pred_pos, true_pos, s=5, label=calibrate_metrics, alpha=0.9, color=color)
            cali_ax.legend(fontsize=10)
        else:
            cali_ax.scatter(pred_pos, true_pos, s=5, alpha=0.9, color=color, label=f"Calibration Plot {remark}")
            if legend:
                cali_ax.legend(fontsize=10)
        cali_ax.plot([0, 1], [0, 1], 'k--', alpha=0.2, linewidth=1.0)
        cali_ax.set_xlabel('Average Predicted Probability', fontsize=12)
        cali_ax.set_ylabel('Empirical Probability', fontsize=12)
        cali_ax.set_xlim(-0.1, 1.1)
        cali_ax.set_ylim(-0.1, 1.1)
        cali_ax.set_title(f"{subtitle}: Calibration Plot", fontsize=14)

    return true_pos, pred_pos, thres

def get_metric_values(true_y, y_prob, thres):
    y_pred = (np.asarray(y_prob) >= thres).astype(int)
    tn, fp, fn, tp = confusion_matrix(true_y, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return {
        'AUC ROC': roc_auc_score(true_y, y_prob),
        'PR-AUC': average_precision_score(true_y, y_prob),
        'Specificity': specificity,
        'Sensitivity': sensitivity,
        'Balanced Acc': 0.5 * (specificity + sensitivity),
        'False Positive Rate': fp / (fp + tn),
        'False Negative Rate': fn / (fn + tp),
    }

def plot_i(m_name="pre_real_all", fold_i=0, chosen_def='fall20', global_axs=None):
    with open((DATA_ROOT + '/splits_5fold_all'), "rb") as f:
        splits = pickle.load(f)
    split = splits[fold_i]["test_fnames"]
    
    # init
    stats = {
        "preds": list(),
        "trues": list()
    }
    
    # fetch output
    with open((DATA_ROOT + '/exp_res/{}/{}_record_{}').format(m_name, m_name, fold_i), "rb") as f:
        records = pickle.load(f)

    # record output
    for j in range(len(split)):
        stats["preds"].append(records["y_preds"][chosen_def][j])
        stats["trues"].append(records["y_trues"][chosen_def][j])
    
    if global_axs is None:
        plt.clf()
        fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(11, 4))
    else:
        axs = global_axs
    pr_ax = axs[1] if len(axs) > 2 else None
    cali_ax = axs[2] if len(axs) > 2 else axs[1]
        
    true_y, y_prob = np.array(stats["trues"]), np.array(stats["preds"])

    # Shuffle with the local seeded generator.
    rng = np.random.default_rng(42)
    idx = rng.permutation(len(true_y))
    true_y, y_prob = true_y[idx], y_prob[idx]

    # plot before calibration
    val_size = 0.2
    _, _, best_threshold = plot_auc_calibrate(true_y[:int(len(true_y)*val_size)], y_prob[:int(len(true_y)*val_size)], auc_ax=None, cali_ax=None)
    true_pos, pred_pos, thres = plot_auc_calibrate(true_y[int(len(true_y)*val_size):], y_prob[int(len(true_y)*val_size):], 
                                                # Plot variant: overlay the uncalibrated ROC curve for the same evaluation subset.
                                                #    auc_ax=axs[0],
                                                   auc_ax=None, 
                                                # Plot variant: overlay the uncalibrated precision-recall curve for the same evaluation subset.
                                                #    pr_ax=pr_ax,
                                                   pr_ax=None,
                                                   cali_ax=cali_ax, all_axs=global_axs, 
                                                   best_threshold=best_threshold, remark='Before Calibration', color="#A2C8E6", 
                                                   legend=fold_i==0, subtitle=label_names[chosen_def])
    before_metrics = get_metric_values(true_y[int(len(true_y)*val_size):], y_prob[int(len(true_y)*val_size):], thres)

    # linear calibrate
    true_pos, pred_pos, thres = plot_auc_calibrate(true_y[:int(len(true_y)*val_size)], y_prob[:int(len(true_y)*val_size)], auc_ax=None, cali_ax=None)
    eps = 1e-6
    safe_true = np.clip(true_pos, eps, 1.0)
    # Calibration variant: pair this condition with the linear KDOQI branch below; both are currently disabled.
    # if chosen_def != 'kdoqi':
    a, b = np.polyfit(pred_pos, np.log(safe_true), 1)
    a = round(a, 2)
    if a < 0.01:
        a = 0.01
    b = round(b, 1)
    y_prob_cali = np.exp(b)*np.exp(a*np.array(y_prob))
    y_prob_cali = np.clip(y_prob_cali, 0, 1)
    print('Calibration Results: A: {}, B: {}'.format(
        round(np.exp(b), 3),
        round(a, 3)
    ))
    # Calibration variant: linear mapping for KDOQI instead of the exponential fit; paired with the condition above.
    # else:
    #     a, b = np.polyfit(pred_pos, safe_true, 1)
    #     a = round(a, 3)
    #     b = round(b, 3)
    #     y_prob_cali = (a*np.array(y_prob)) + b
    #     y_prob_cali = np.clip(y_prob_cali, 0, 1)
    #     print('Calibration Results: A: {}, B: {}'.format(
    #         round(a, 3),
    #         round(b, 3)
    #     ))

    # plot after calibration
    _, _, best_threshold = plot_auc_calibrate(true_y[:int(len(true_y)*val_size)], y_prob_cali[:int(len(true_y)*val_size)], auc_ax=None, cali_ax=None)
    true_pos, pred_pos, thres = plot_auc_calibrate(true_y[int(len(true_y)*val_size):], y_prob_cali[int(len(true_y)*val_size):], auc_ax=axs[0], cali_ax=cali_ax, pr_ax=pr_ax, 
                                                   best_threshold=best_threshold, remark='After Calibration', color="#F6A5B4", 
                                                   all_axs=global_axs, legend=fold_i==0, subtitle=label_names[chosen_def])
    after_metrics = get_metric_values(true_y[int(len(true_y)*val_size):], y_prob_cali[int(len(true_y)*val_size):], thres)

    if axs is None:
        plt.suptitle(label_names[chosen_def], fontsize=16)
        plt.tight_layout()
        plt.show()

    return {'Before Calibration': before_metrics, 'After Calibration': after_metrics}


In [ ]:

# Analysis variants: evaluate individual IDH definitions using the same fold and calibration procedure.
# plot_i(m_name="pre_real_all_adjust_ehr", fold_i=0, chosen_def='fall20')
# plot_i(m_name="pre_real_all_adjust_ehr", fold_i=0, chosen_def='nadir90')
# plot_i(m_name="pre_real_all_adjust_ehr", fold_i=0, chosen_def='hemo')
# plot_i(m_name="pre_real_all_adjust_ehr", fold_i=0, chosen_def='kdoqi')

# concat all scatter together
chosen_def='nadir90'
plt.clf()
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(8, 4))
for i in range(5):
    plot_i(m_name="pre_real_all_adjust_ehr", fold_i=i, chosen_def=chosen_def, global_axs=axs)

plt.suptitle(label_names[chosen_def], fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
plt.clf()
fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(12, 12))

c_i = 0
for chosen_def in ['fall20', 'nadir90', 'hemo']:
    for i in range(5):
        plot_i(m_name="pre_real_all_adjust_ehr", fold_i=i, chosen_def=chosen_def, global_axs=[axs[0, c_i], axs[1, c_i], axs[2, c_i]])
    # Average the five fold curves on a shared FPR/recall grid.
    grid = np.linspace(0, 1, 1001)
    for ax in [axs[0, c_i], axs[1, c_i]]:
        labels = list(dict.fromkeys(line.get_label() for line in ax.lines if not line.get_label().startswith('_')))
        for label in labels:
            lines = [line for line in ax.lines if line.get_label() == label]
            mean_y = np.mean([np.interp(grid, line.get_xdata(), line.get_ydata()) for line in lines], axis=0)
            color = lines[0].get_color()
            for line in lines:
                line.remove()
            ax.plot(grid, mean_y, color=color, label=f'{label}')
        ax.legend(fontsize=10)
    c_i += 1


plt.tight_layout()
plt.savefig(
    (FIGURE_ROOT + '/calibration_plot.pdf'),
    format="pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
fold_metrics = []
for i in range(5):
    fold_metrics.append(plot_i(m_name="pre_real_all_adjust_ehr", fold_i=i, chosen_def='nadir100'))

for stage in ['Before Calibration', 'After Calibration']:
    print(f'\n{stage} (mean ± std across 5 folds)')
    for metric in fold_metrics[0][stage]:
        values = [fold[stage][metric] for fold in fold_metrics]
        print(f'{metric}: {np.mean(values):.3f}$\pm${np.std(values, ddof=1):.3f}')
